In [1]:
import pandas as pd
from pathlib import Path
import yaml

def find_project_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "pyproject.toml").exists() or (parent / ".git").exists():
            return parent
    raise RuntimeError("Project root not found (pyproject.toml/.git)")

PROJECT_ROOT = find_project_root()
CONFIG_ROOT = PROJECT_ROOT / "config"
dataset_cfg_path = CONFIG_ROOT / "datasets" / "dataset_config.yml"

with open(dataset_cfg_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

ds = cfg["datasets"]["microsoft_security_incident"]
train_rel = ds["paths"]["train"]

test_rel= ds["paths"]["test"]

csv_params = ds.get("csv_params", {})

train_path = PROJECT_ROOT / train_rel
test_path = PROJECT_ROOT / test_rel

In [2]:
df_head = pd.read_csv(
    train_path,
    sep=csv_params.get("sep", ","),
    encoding=csv_params.get("encoding", "utf-8"),
    decimal=csv_params.get("decimal", "."),
    low_memory=csv_params.get("low_memory", False),
    nrows=10
)
df_head

,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,IncidentGrade,...,ResourceType,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City
0,180388628218,0,612,123247,2024-06-04T06:05:15.000Z,7,6,InitialAccess,NaN,TruePositive,...,NaN,NaN,5,66,NaN,NaN,NaN,31,6,3
1,455266534868,88,326,210035,2024-06-14T03:01:25.000Z,58,43,Exfiltration,NaN,FalsePositive,...,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
2,1056561957389,809,58352,712507,2024-06-13T04:52:55.000Z,423,298,InitialAccess,T1189,FalsePositive,...,NaN,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630
3,1279900258736,92,32992,774301,2024-06-10T16:39:36.000Z,2,2,CommandAndControl,NaN,BenignPositive,...,NaN,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630
4,214748368522,148,4359,188041,2024-06-15T01:08:07.000Z,9,74,Execution,NaN,TruePositive,...,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
5,1322849927433,11,417400,825450,2024-06-10T13:30:56.000Z,0,0,InitialAccess,T1078;T1078.004,FalsePositive,...,NaN,NaN,5,66,NaN,NaN,NaN,8,6,3
6,163208760309,522,566,705663,2024-06-14T23:19:45.000Z,2,2,CommandAndControl,NaN,BenignPositive,...,NaN,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630
7,1400159339557,125,38679,47423,2024-06-06T13:39:23.000Z,313,3919,Exfiltration,NaN,BenignPositive,...,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
8,1219770713645,21,414,197969,2024-06-09T10:21:29.000Z,3,4,SuspiciousActivity,NaN,BenignPositive,...,NaN,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630
9,1073741827836,72,70,831157,2024-06-08T02:08:01.000Z,4,3,InitialAccess,NaN,TruePositive,...,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630


In [3]:
import duckdb

ORG_ID = 11
INCIDENT_ID = 417400

query = f"""
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT Id) AS n_distinct_id,
  COUNT(DISTINCT AlertId) AS n_distinct_alert,
  COUNT(DISTINCT CAST(Timestamp AS VARCHAR)) AS n_distinct_ts
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
WHERE OrgId = {ORG_ID} AND IncidentId = {INCIDENT_ID}
"""
duckdb.query(query).to_df()

,n_rows,n_distinct_id,n_distinct_alert,n_distinct_ts
0,4,1,1,1


In [4]:
query = f"""
SELECT DISTINCT
  *
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
WHERE OrgId = {ORG_ID} AND IncidentId = {INCIDENT_ID}
ORDER BY Timestamp
"""
incident_df = duckdb.query(query).to_df()

import pandas as pd
pd.set_option("display.max_columns", None)

incident_df
#incident_df.iloc[0].to_frame("value")

,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,IncidentGrade,ActionGrouped,ActionGranular,EntityType,EvidenceRole,DeviceId,Sha256,IpAddress,Url,AccountSid,AccountUpn,AccountObjectId,AccountName,DeviceName,NetworkMessageId,EmailClusterId,RegistryKey,RegistryValueName,RegistryValueData,ApplicationId,ApplicationName,OAuthApplicationId,ThreatFamily,FileName,FolderPath,ResourceIdName,ResourceType,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City
0,1322849927433,11,417400,825450,2024-06-10 15:30:56+02:00,0,0,InitialAccess,T1078;T1078.004,FalsePositive,None,None,CloudLogonSession,Related,98799,138268,360606,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,None,289573,117668,3586,None,None,5,66,None,None,None,242,1445,10630
1,1322849927433,11,417400,825450,2024-06-10 15:30:56+02:00,0,0,InitialAccess,T1078;T1078.004,FalsePositive,None,None,CloudLogonRequest,Related,98799,138268,360606,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,None,289573,117668,3586,None,None,5,66,None,None,None,242,1445,10630
2,1322849927433,11,417400,825450,2024-06-10 15:30:56+02:00,0,0,InitialAccess,T1078;T1078.004,FalsePositive,None,None,User,Impacted,98799,138268,360606,160396,28670,48804,29043,32324,153085,529644,NaN,1631,635,860,2251,3421,881,None,289573,117668,3586,None,None,5,66,None,None,None,242,1445,10630
3,1322849927433,11,417400,825450,2024-06-10 15:30:56+02:00,0,0,InitialAccess,T1078;T1078.004,FalsePositive,None,None,Ip,Related,98799,138268,30410,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,None,289573,117668,3586,None,None,5,66,None,None,None,8,6,3


In [3]:
import duckdb

delim = csv_params.get("sep", ",")

base = f"read_csv_auto('{train_path}', delim='{delim}')"

base_test = f"read_csv_auto('{test_path}', delim='{delim}')"

In [5]:
schema_df = duckdb.query(f"DESCRIBE SELECT * FROM {base}").to_df()
#schema_df, len(schema_df)
schema_df[["column_name", "column_type"]]

,column_name,column_type
0,Id,BIGINT
1,OrgId,BIGINT
2,IncidentId,BIGINT
3,AlertId,BIGINT
4,Timestamp,TIMESTAMP WITH TIME ZONE
5,DetectorId,BIGINT
6,AlertTitle,BIGINT
7,Category,VARCHAR
8,MitreTechniques,VARCHAR
9,IncidentGrade,VARCHAR


In [4]:
schema_df = duckdb.query(f"DESCRIBE SELECT * FROM {base_test}").to_df()
schema_df, len(schema_df)

(           column_name               column_type null   key default extra
 0                   Id                    BIGINT  YES  None    None  None
 1                OrgId                    BIGINT  YES  None    None  None
 2           IncidentId                    BIGINT  YES  None    None  None
 3              AlertId                    BIGINT  YES  None    None  None
 4            Timestamp  TIMESTAMP WITH TIME ZONE  YES  None    None  None
 5           DetectorId                    BIGINT  YES  None    None  None
 6           AlertTitle                    BIGINT  YES  None    None  None
 7             Category                   VARCHAR  YES  None    None  None
 8      MitreTechniques                   VARCHAR  YES  None    None  None
 9        IncidentGrade                   VARCHAR  YES  None    None  None
 10       ActionGrouped                   VARCHAR  YES  None    None  None
 11      ActionGranular                   VARCHAR  YES  None    None  None
 12          EntityType  

In [4]:
keys_df = duckdb.query(f"""
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT Id) AS n_distinct_id,
  COUNT(DISTINCT (OrgId, IncidentId)) AS n_distinct_incident,
  COUNT(DISTINCT (OrgId, AlertId)) AS n_distinct_alert
FROM {base}
""").to_df()
keys_df

,n_rows,n_distinct_id,n_distinct_incident,n_distinct_alert
0,9516837,730778,730778,1265644


In [6]:
duckdb.query(f"""
SELECT
  COUNT(*) AS n_incidents,
  SUM(CASE WHEN n_grades > 1 THEN 1 ELSE 0 END) AS incidents_with_multiple_grades
FROM (
  SELECT OrgId, IncidentId, COUNT(DISTINCT IncidentGrade) AS n_grades
  FROM {base}
  GROUP BY OrgId, IncidentId
)
""").to_df()

,n_incidents,incidents_with_multiple_grades
0,730778,0.0


In [5]:
evidence_per_alert = duckdb.query(f"""
SELECT
  approx_quantile(cnt, 0.50) AS p50,
  approx_quantile(cnt, 0.90) AS p90,
  approx_quantile(cnt, 0.99) AS p99,
  max(cnt) AS max_evidences_per_alert
FROM (
  SELECT OrgId, AlertId, COUNT(*) AS cnt
  FROM {base}
  GROUP BY OrgId, AlertId
)
""").to_df()
evidence_per_alert

,p50,p90,p99,max_evidences_per_alert
0,4,12,53,11292


In [6]:
import pandas as pd
import duckdb

schema_df = duckdb.query(f"DESCRIBE SELECT * FROM {base}").to_df()
cols = schema_df["column_name"].tolist()

# Construye SELECT con % non-null por columna
parts = []
for c in cols:
    parts.append(f"(COUNT({c}) * 100.0 / COUNT(*)) AS '{c}'")

sql = "SELECT " + ",\n  ".join(parts) + f"\nFROM {base}"
completeness_wide = duckdb.query(sql).to_df()

# Pásalo a formato largo (Colonna, Completezza_%)
completeness = (
    completeness_wide.T.reset_index()
    .rename(columns={"index":"Colonna", 0:"Completezza_%"} )
)
completeness

,Colonna,Completezza_%
0,Id,100.000000
1,OrgId,100.000000
2,IncidentId,100.000000
3,AlertId,100.000000
4,Timestamp,100.000000
5,DetectorId,100.000000
6,AlertTitle,100.000000
7,Category,100.000000
8,MitreTechniques,42.539880
9,IncidentGrade,99.460535


In [7]:
duckdb.query(f"""
SELECT
  COUNT(*) AS n_rows,
  SUM(CASE WHEN EntityType IS NULL THEN 1 ELSE 0 END) AS n_null_entitytype
FROM {base}
""").to_df()

,n_rows,n_null_entitytype
0,9516837,0.0
